# FinBERT financial sentiment

This notebook evaluates a pretrained checkpoint first and optionally fine-tunes it.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.text_utils import load_data, metrics_table, set_seed, summarize_splits
from utils.transformer_utils import build_transformer_trainer, evaluate_transformer_checkpoint

set_seed(42)

I0000 00:00:1777752262.475677  260684 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777752262.505505  260684 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777752263.339178  260684 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777752263.339454  260684 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will 

In [2]:
BASELINE_MODEL_NAME = "ProsusAI/finbert"
MODEL_LABEL_ORDER = ["positive", "negative", "neutral"]
OUTPUT_DIR = ROOT / "artifacts" / "finbert"

df = load_data()
display(summarize_splits(df))
BASELINE_MODEL_NAME

,rows,mean_words,negative,neutral,positive
split,,,,,
test,23566,17.30,5817,7306,10443
train,77589,18.01,18856,25403,33330
validation,23567,17.36,5722,7261,10584


'ProsusAI/finbert'

In [3]:
baseline_results = evaluate_transformer_checkpoint(BASELINE_MODEL_NAME, MODEL_LABEL_ORDER, df, split="test", max_length=128, batch_size=32)
display("Off-the-shelf baseline")
display(metrics_table(baseline_results))
display(baseline_results["report"])
display(baseline_results["confusion_matrix"])
baseline_results["predictions"].head()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

evaluating ProsusAI/finbert:   0%|          | 0/737 [00:00<?, ?it/s]

'Off-the-shelf baseline'

,metric,value
0,accuracy,0.600526
1,macro_precision,0.663223
2,macro_recall,0.620886
3,macro_f1,0.606948
4,weighted_f1,0.602276


,precision,recall,f1-score,support
negative,0.700381,0.599966,0.646296,5817.000000
neutral,0.458330,0.804681,0.584016,7306.000000
positive,0.830959,0.458010,0.590530,10443.000000
accuracy,0.600526,0.600526,0.600526,0.600526
macro avg,0.663223,0.620886,0.606948,23566.000000
weighted avg,0.683204,0.600526,0.602276,23566.000000


,negative,neutral,positive
negative,3490,2006,321
neutral,775,5879,652
positive,718,4942,4783


,text,label,prediction
0,hqge ltnc hbrm enzc eeenf halb azfl maxd mmex ...,positive,neutral
1,econx november nonfarm private payrolls k vs k...,positive,neutral
2,regulatory news the nomination committee of cy...,neutral,neutral
3,amazon labor union is now seeking to represent...,positive,neutral
4,greene king s third quarter sales boosted by f...,positive,positive


In [4]:
trainer, datasets, tokenizer = build_transformer_trainer(
    df,
    model_name=BASELINE_MODEL_NAME,
    model_label_order=MODEL_LABEL_ORDER,
    output_dir=str(OUTPUT_DIR),
    max_length=128,
    epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
)

train_result = trainer.train()
fine_tuned_test_metrics = trainer.evaluate(datasets["test"])
best_checkpoint = trainer.state.best_model_checkpoint or str(OUTPUT_DIR)
display("Fine-tuned checkpoint metrics")
display(metrics_table(train_result.metrics))
display(metrics_table(fine_tuned_test_metrics))
display(best_checkpoint)
final_results = evaluate_transformer_checkpoint(best_checkpoint, MODEL_LABEL_ORDER, df, split="test", max_length=128, batch_size=32)
display(metrics_table(final_results))
display(final_results["report"])
display(final_results["confusion_matrix"])
final_results["predictions"].head()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/77589 [00:00<?, ? examples/s]

Map:   0%|          | 0/23567 [00:00<?, ? examples/s]

Map:   0%|          | 0/23566 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,0.493853,0.306764,0.888785,0.886305,0.882719,0.884449,0.888647
2,0.247978,0.316512,0.901515,0.898214,0.896299,0.897115,0.901396


...

evaluating /home/automac/Documents/Estudios/Semestre3/ProyectoDeGrado/artifacts/finbert/checkpoint-9700:   0%|…

,metric,value
0,accuracy,0.902147
1,macro_precision,0.899545
2,macro_recall,0.898172
3,macro_f1,0.898700
4,weighted_f1,0.902050


,precision,recall,f1-score,support
negative,0.879479,0.894447,0.886900,5817.000000
neutral,0.908754,0.873802,0.890936,7306.000000
positive,0.910400,0.926266,0.918265,10443.000000
accuracy,0.902147,0.902147,0.902147,0.902147
macro avg,0.899545,0.898172,0.898700,23566.000000
weighted avg,0.902257,0.902147,0.902050,23566.000000


,negative,neutral,positive
negative,5203,219,395
neutral,365,6384,557
positive,348,422,9673


,text,label,prediction
0,hqge ltnc hbrm enzc eeenf halb azfl maxd mmex ...,positive,positive
1,econx november nonfarm private payrolls k vs k...,positive,positive
2,regulatory news the nomination committee of cy...,neutral,neutral
3,amazon labor union is now seeking to represent...,positive,neutral
4,greene king s third quarter sales boosted by f...,positive,positive
